# Counterfactual Analysis

In [1]:
import warnings
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random

from catboost import CatBoostClassifier

from pathlib import Path
import shutil
import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig


%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
from omnixai.data.tabular import Tabular
from omnixai.preprocessing.tabular import TabularTransform
from omnixai.explainers.tabular import MACEExplainer
from omnixai.explainers.tabular import TabularExplainer

## Load Counterfactuals Module

In [3]:
from utils2 import macecf as mcf

## Read Config File

In [4]:
config_path = Path(r'experiments')
config_filename =  "bin_mace_cf_dev.yml"
config_dict = ymlconfig.load_config(config_path / config_filename)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - Counterfactual Analysis (Dev)',
  'classification_type': 'binary',
  'stage': 'counterfactuals-mace',
  'tag': 'development',
  'verbosity': 1,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'code': 'catboost', 'name': 'CatBoost'},
 'explainability': {'ksplit_trained_model_results_file': 'binary\\explainability\\catboost\\final\\catboost_ksplit_trained_models.joblib',
  'rundate': '2026-03-18',
  'tag': 'final'},
 'mace': {'cf_features': {'actionable': 'INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI',
   'categorical': 'INSULIN,HPN,PAOD,DSLPDMIA,CKD,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR',
   'progressive': 'none'}}}

#### Set output directory

In [5]:
outputdir = config_path /  config.experiment.classification_type /  config.experiment.stage / config.model.code / config.experiment.tag 
outputdir.mkdir(parents=True, exist_ok=True)
print(outputdir)

experiments\binary\counterfactuals-mace\catboost\development


#### Copy config file to output directory

In [6]:
source = config_path / config_filename
destination = outputdir / config_filename
shutil.copy(source, destination)

WindowsPath('experiments/binary/counterfactuals-mace/catboost/development/bin_mace_cf_dev.yml')

## Data Loading

In [7]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
X = dfdpn[data_cols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 40), (190,))

In [8]:
dfXy = pd.concat([X, y], axis=1)
X.shape, y.shape, dfXy.shape

((190, 40), (190,), (190, 41))

### Define custom column lists

In [9]:
features_to_vary = dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list()
continuous_cols = dfXy.columns.difference(D.categorical_cols+['SEX', 'Confirmed_Binary_DPN']).to_list()
print('features to vary columns:\n', len(features_to_vary), features_to_vary)
print('categorical columns:\n', len(D.categorical_cols), D.categorical_cols)
print('continuous_columns:\n', len(continuous_cols), continuous_cols)

features to vary columns:
 39 ['AGE', 'SUBJ', 'DM_DUR', 'INSULIN', 'HBA1C', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR', 'MNSI', 'SSA_L', 'SSC_L', 'SPSA_L', 'SPSC_L', 'MCV_L', 'DL_L', 'CMAPANK_L', 'CMAPKNE_L', 'FWAVE_L', 'SSA_R', 'SSC_R', 'SPSA_R', 'SPSC_R', 'MCV_R', 'DL_R', 'CMAPANK_R', 'CMAPKNE_R', 'FWAVE_R', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM', 'NS', 'CAS']
categorical columns:
 12 ['SEX', 'SUBJ', 'INSULIN', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'GBS', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR']
continuous_columns:
 28 ['AGE', 'CAS', 'CMAPANK_L', 'CMAPANK_R', 'CMAPKNE_L', 'CMAPKNE_R', 'DL_L', 'DL_R', 'DM_DUR', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'FWAVE_L', 'FWAVE_R', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM', 'HBA1C', 'MCV_L', 'MCV_R', 'MNSI', 'NS', 'SPSA_L', 'SPSA_R', 'SPSC_L', 'SPSC_R', 'SSA_L', 'SSA_R', 'SSC_L', 'SSC_R']


## Load trained model splits from Explainability Stage

### Load Models and Check Rundate and Tag

In [10]:
ksplit_trained_models = joblib.load(config_path / config.explainability.ksplit_trained_model_results_file)
assert ksplit_trained_models['rundate'] == config.explainability.rundate, f"{ksplit_trained_models['rundate']} != {config.explainability.rundate}"
assert ksplit_trained_models['tag'] == config.explainability.tag
print('rundate:', ksplit_trained_models['rundate'])
print('tag:', ksplit_trained_models['tag'])
ksplit_trained_models['summary']

rundate: 2026-03-18
tag: final


,0,1,2,mean,std
youden,0.909,0.727,0.784,0.807,0.076
roc_auc,0.983,0.986,0.944,0.971,0.019


In [11]:
split_results = ksplit_trained_models['results']
split_results[0]

{'model': CatBoostClassifier(depth=6, eval_metric='AUC', iterations=391, l2_leaf_reg=8.462943990782671, learning_rate=0.02069186296126172, loss_function='Logloss', random_state=42, verbose=0),
 'X_train':      SEX   AGE  SUBJ  DM_DUR  INSULIN  HBA1C  HPN  PAOD  DSLPDMIA  CKD  GBS  \
 0      1  64.0     1     7.0      1.0  15.00    0     0         0    0    0   
 1      0  59.0     1     1.0      0.0   5.60    1     0         0    0    0   
 5      0  20.0     1     2.0      1.0   7.80    0     0         0    0    0   
 6      0  69.0     0     0.0      0.0   8.00    1     0         1    0    0   
 7      0  60.0     0     2.0      0.0   5.80    1     0         0    0    0   
 8      1  62.0     0     0.0      1.0  14.36    0     0         0    0    0   
 9      0  44.0     1    17.0      0.0   7.01    0     0         0    0    0   
 10     0  70.0     0    10.0      0.0   6.40    1     0         0    1    0   
 11     1  61.0     1     4.0      0.0   8.30    0     0         1    1    0

### Extract Train and Test Sets

In [12]:
# Choose a model split
midx = 0

# Create output directory for this Model split
split_output_dir = outputdir / f'split{midx}'
split_output_dir.mkdir(parents=True, exist_ok=True)

# Extract saved variables from split
threshold = split_results[midx]['threshold']
best_params = split_results[midx]['best_params']

# Extract test set
X_test = split_results[midx]['X_test']
y_test = split_results[midx]['y_test']
dfXy_test = pd.concat([X_test, y_test], axis=1)
X_test.shape, y_test.shape, dfXy_test.shape, threshold

((64, 40), (64,), (64, 41), 0.6352084424142594)

In [13]:
X_train = split_results[midx]['X_train']
X_test = split_results[midx]['X_test']

# convert categorical columns in X_train - needed in CatBoost for use in DiCE
X_train[D.categorical_cols] = X_train[D.categorical_cols].astype(str)
X_test[D.categorical_cols] = X_test[D.categorical_cols].astype(str)

y_train = split_results[midx]['y_train']
y_test = split_results[midx]['y_test']

## Prepare Explainer

### Refit the model 

In [14]:
### Refit model so we can set cat_features (needed in DiCE)
model=  CatBoostClassifier(**best_params, 
                        #    cat_features=D.categorical_cols, 
                           verbose=0
                           ).fit(X_train, y_train)

###  Wrap model for threshold

In [15]:
# Wrap model so we can use a custom threshold
wrapped_model = mcf.CatBoostWrapper(model, threshold)
mcf.test_wrapped_model(model, wrapped_model, X_test, y_test, threshold, verbosity=1)

Confusion Matrix at default threshold (0.5): /n[[20  0]
 [ 4 40]]
Confusion Matrix at custom threshold (0.64): /n([[20  0]
 [ 4 40]]):
Rows with different predictions at thresholds: 


,SEX,AGE,SUBJ,DM_DUR,INSULIN,HBA1C,HPN,PAOD,DSLPDMIA,CKD,GBS,DEC_VS,DEC_PPS,DEC_LTS,DEC_AR,MNSI,SSA_L,SSC_L,SPSA_L,SPSC_L,MCV_L,DL_L,CMAPANK_L,CMAPKNE_L,FWAVE_L,SSA_R,SSC_R,SPSA_R,SPSC_R,MCV_R,DL_R,CMAPANK_R,CMAPKNE_R,FWAVE_R,FEET_MEAN_ESC,FEET_PCT_ASYM,HAND_MEAN_ESC,HAND_PCT_ASYM,NS,CAS,pred_0.50,pred_0.64,pred_proba_0.50,pred_proba_0.64


## MACE Explainer Object Setup

In [16]:
features_to_vary = config.mace.cf_features.actionable.split(',')
print(features_to_vary)
# continuous_cols = dfXy.columns.difference(D.categorical_cols+['Confirmed_Binary_DPN']).to_list()

['INSULIN', 'HBA1C', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR', 'MNSI']


In [17]:
categorical_features = config.mace.cf_features.categorical.split(',')
print(categorical_features)

['INSULIN', 'HPN', 'PAOD', 'DSLPDMIA', 'CKD', 'DEC_VS', 'DEC_PPS', 'DEC_LTS', 'DEC_AR']


In [18]:
features_to_ignore = dfXy.columns.difference(features_to_vary+['Confirmed_Binary_DPN']).to_list()
print(features_to_ignore)

['AGE', 'CAS', 'CMAPANK_L', 'CMAPANK_R', 'CMAPKNE_L', 'CMAPKNE_R', 'DL_L', 'DL_R', 'DM_DUR', 'FEET_MEAN_ESC', 'FEET_PCT_ASYM', 'FWAVE_L', 'FWAVE_R', 'GBS', 'HAND_MEAN_ESC', 'HAND_PCT_ASYM', 'MCV_L', 'MCV_R', 'NS', 'SEX', 'SPSA_L', 'SPSA_R', 'SPSC_L', 'SPSC_R', 'SSA_L', 'SSA_R', 'SSC_L', 'SSC_R', 'SUBJ']


## Trial 1

In [ ]:
# ── 1. Build test tabular (reattach target for OmniXAI) ──────────────────────
test_df = X_test.copy()
test_df['Confirmed_Binary_DPN'] = y_test.values

test_tabular = Tabular(
    test_df,
    categorical_columns=categorical_features,
    target_column='Confirmed_Binary_DPN'
)

In [ ]:

# ── 2. Fit transformer on FULL data (same one used for tabular_data) ─────────
tabular_data = Tabular(
    dfXy,
    categorical_columns=categorical_features,
    target_column='Confirmed_Binary_DPN'
)
transformer = TabularTransform().fit(tabular_data)
class_names = transformer.class_names   # e.g. ['No DPN', 'DPN']

In [ ]:

# ── 3. Build predict function for wrapped CatBoost model ─────────────────────
# TabularExplainer needs a function that:
#   - receives a numpy array (transformed/encoded)
#   - returns class probabilities of shape (n, num_classes)
def predict_fn(x: np.ndarray) -> np.ndarray:
    # Invert OmniXAI's encoding back to a DataFrame
    df = transformer.invert(x)
    return wrapped_model.predict_proba(df)   # uses your custom threshold wrapper

In [ ]:
# ── 4. Select test instances to explain ──────────────────────────────────────
# Remove target column before passing to explainer
# Adjust the slice [0:5] to whichever patient indices you want
test_instances = test_tabular.remove_target_column()[0:5]

# ── 5. Initialize TabularExplainer ───────────────────────────────────────────
explainer = TabularExplainer(
    explainers=["shap", "mace"],          # SHAP for importance, MACE for CFs
    mode="classification",
    data=tabular_data,                    # full dataset — MACE uses this for candidate search
    # model=predict_fn,
    model=wrapped_model,
    preprocess=transformer.transform,     # OmniXAI applies this before calling predict_fn
    params={
        "mace": {
            "ignored_features": features_to_ignore,   # from your config
            "method": "gld",                          # gradient-less — correct for CatBoost
        },
        # "shap": {
        #     "nsamples": 100,                          # background samples for SHAP
        # },
    }
)

In [ ]:
# Full data — for fitting the transformer and MACE explainer
tabular_data = Tabular(
    dfXy,
    categorical_columns=categorical_features,
    target_column='Confirmed_Binary_DPN'
)
transformer = TabularTransform().fit(tabular_data)
class_names = transformer.class_names

# Test split — for generating counterfactuals
test_df = X_test.copy()
test_df['Confirmed_Binary_DPN'] = y_test.values

test_tabular = Tabular(
    test_df,
    categorical_columns=categorical_features,
    target_column='Confirmed_Binary_DPN'
)

# Strip target when passing to explainer
test_instances = test_tabular.remove_target_column()

In [ ]:
# ── 6. Local explanations (per patient) ──────────────────────────────────────
local_explanations = explainer.explain(test_instances)

In [ ]:

# ── 7. Global explanations (dataset-level) ───────────────────────────────────
global_explanations = explainer.explain_global(
    params={"pdp": {"target_class": 1}}   # class 1 = DPN positive
)

---

### Take Two

In [19]:
tabular_data = Tabular(
    dfXy,
    categorical_columns=categorical_features,
    target_column='Confirmed_Binary_DPN'
)

# ── 1. Fit transformer on full labelled data ─────────────────────────────────
transformer = TabularTransform().fit(tabular_data)
class_names = transformer.class_names

In [20]:
# ── 2. Transform → split → invert back to Tabular ────────────────────────────
# This is the pattern OmniXAI expects — train_data and test_data
# will NOT have the target column, which is what MACE requires internally

x = transformer.transform(tabular_data)   # full data as numpy array

# Recreate train/test split indices from your existing X_train/X_test
# to keep consistency with your CatBoost model's split
train_indices = X_train.index
test_indices  = X_test.index

x_all    = x[:, :-1]    # features only (drop last col = target)
x_labels = x[:, -1]     # labels only

# Map your existing split back to transformed numpy arrays
all_idx   = dfXy.index.tolist()
train_pos = [all_idx.index(i) for i in train_indices]
test_pos  = [all_idx.index(i) for i in test_indices]

x_train = x_all[train_pos]
x_test  = x_all[test_pos]

# Invert back to Tabular (no target column — this is what MACE needs)
train_data = transformer.invert(x_train)
test_data  = transformer.invert(x_test)

In [21]:
train_indices

Index([  0,   1,   5,   6,   7,   8,   9,  10,  11,  12,
       ...
       179, 180, 181, 182, 183, 185, 186, 187, 188, 189],
      dtype='int64', length=126)

In [22]:
test_indices

Index([  2,   3,   4,  13,  15,  17,  19,  20,  23,  27,  28,  29,  31,  37,
        39,  40,  44,  46,  52,  53,  55,  56,  59,  61,  64,  65,  67,  70,
        71,  72,  74,  78,  83,  84,  86,  87,  89,  91,  92,  98,  99, 101,
       102, 103, 106, 107, 115, 124, 128, 134, 135, 141, 147, 151, 152, 153,
       156, 158, 159, 161, 162, 174, 176, 184],
      dtype='int64')

In [23]:
x_train.shape, x_test.shape

((126, 49), (64, 49))

In [24]:
# ── 3. Build predict function ─────────────────────────────────────────────────
# def predict_fn(x: np.ndarray) -> np.ndarray:
#     # x arrives as transformed numpy array — invert to DataFrame for CatBoost
#     df = transformer.invert(x).to_pd()
#     return wrapped_model.predict_proba(df)   # shape (n, 2)

---

In [32]:
# Step 1 — check actual class sizes first
train_pd = X_train.copy()
train_pd['Confirmed_Binary_DPN'] = y_train.values

print("Class counts in training data:")
print(train_pd['Confirmed_Binary_DPN'].value_counts())

# Step 2 — set n_per_class to the MINORITY class size
minority_count = train_pd['Confirmed_Binary_DPN'].value_counts().min()
print(f"\nMinority class count: {minority_count}")
print(f"Sampling {minority_count} from each class")

# Step 3 — sample exactly minority_count from each class
class_0 = train_pd[train_pd['Confirmed_Binary_DPN'] == 0].sample(
    n=minority_count, random_state=42
)
class_1 = train_pd[train_pd['Confirmed_Binary_DPN'] == 1].sample(
    n=minority_count, random_state=42
)

balanced_df = pd.concat([class_0, class_1]).reset_index(drop=True)

print("\nBalanced class distribution:")
print(balanced_df['Confirmed_Binary_DPN'].value_counts())
print(f"Total samples: {len(balanced_df)}")

Class counts in training data:
Confirmed_Binary_DPN
1    86
0    40
Name: count, dtype: int64

Minority class count: 40
Sampling 40 from each class

Balanced class distribution:
Confirmed_Binary_DPN
0    40
1    40
Name: count, dtype: int64
Total samples: 80


In [33]:
# ── 4. Initialize TabularExplainer ───────────────────────────────────────────
explainer = TabularExplainer(
    explainers=["shap", "mace"],
    mode="classification",
    data=train_data,                          # ← target-free Tabular
    model=wrapped_model,
    preprocess=lambda z: transformer.transform(z),
    params={
        "shap": {"nsamples": 100},
        "mace": {
            "ignored_features": ['AGE', 'DM_DUR'], #features_to_ignore,
            "method": "gld",                  # gradient-less — required for CatBoost
            "num_neighbors": 500,
        },
    }
)

In [34]:
# ── 5. Select test instances to explain ──────────────────────────────────────
test_instances = test_data[[22,45,49]]              # ← already target-free, no .remove_target_column()

In [35]:
# ── 6. Run local and global explanations ─────────────────────────────────────
local_explanations  = explainer.explain(X=test_instances)
# global_explanations = explainer.explain_global(
#     params={"pdp": {"features": features_to_vary}}
# )

100%|██████████| 3/3 [00:03<00:00,  1.32s/it]


KeyError: 'Explainer mace -- 0'

In [ ]:
### Refit model so we can set cat_features (needed in DiCE)
model=  CatBoostClassifier(**best_params, 
                           cat_features=D.categorical_cols, 
                           verbose=0
                           ).fit(X_train, y_train)

In [ ]:
# Wrap model so we can use a custom threshold
wrapped_model = mcf.CatBoostWrapper(model, threshold)
mcf.test_wrapped_model(model, wrapped_model, X_test, y_test, threshold, verbosity=1)

In [ ]:
train, test, train_labels, test_labels = \
    sklearn.model_selection.train_test_split(x[:, :-1], x[:, -1], train_size=0.80)
# Train an XGBoost model (the last column of `x` is the label column after transformation)
model = xgboost.XGBClassifier(n_estimators=300, max_depth=5)
model.fit(train, train_labels)

In [ ]:

import sklearn
from omnixai.preprocessing.tabular import TabularTransform
# Data preprocessing
transformer = TabularTransform().fit(tabular_data)
class_names = transformer.class_names
x = transformer.transform(tabular_data)
# Split into training and test datasets
train, test, train_labels, test_labels = \
    sklearn.model_selection.train_test_split(x[:, :-1], x[:, -1], train_size=0.80)
# Train an XGBoost model (the last column of `x` is the label column after transformation)

model = xgboost.XGBClassifier(n_estimators=300, max_depth=5)
model.fit(train, train_labels)
# Convert the transformed data back to Tabular instances
train_data = transformer.invert(train)
test_data = transformer.invert(test)

In [ ]:
from omnixai.explainers.tabular import TabularExplainer
# Initialize a TabularExplainer
explainer = TabularExplainer(
  explainers=["lime", "shap", "mace", "pdp", "ale"], # The explainers to apply
  mode="classification",                             # The task type
  data=X_train,                                   # The data for initializing the explainers
  model=wrapped_model,                                       # The ML model to explain
  # preprocess=lambda z: transformer.transform(z),     # Converts raw features into the model inputs
  params={
     "mace": {"ignored_features": features_to_ignore}
  }                                                  
)

###  Define Global Permitted Range

In [ ]:
global_permitted_range = cf.get_global_permitted_range(
    dfXy, continuous_cols, config, midx, verbosity=1, savedir=split_output_dir)

### Define DiCE Explainer Object

In [ ]:
d = dice_ml.Data(dataframe=dfXy_test, # use only the test set
                 continuous_features=continuous_cols,                  
                 categorical_features=D.categorical_cols,
                 permitted_range = global_permitted_range, 
                 outcome_name='Confirmed_Binary_DPN')

m = dice_ml.Model(model=wrapped_model, backend="sklearn", model_type="classifier")
dexp = dice_ml.Dice(d, m, method=config.dice.method)

## Get Global Importances

In [ ]:
cf.get_global_importance(dexp, D, X_test, config, midx, 
                         features_to_vary, threshold, global_permitted_range,
                         highlight_features=cf.actionable_cols, 
                         filename_suffix="", savedir=split_output_dir)

## Local Counterfactual Analysis

##### Local Permitted Range for a Patient 

In [ ]:
pidx = 67
split_instance_output_dir = split_output_dir / f'{str(pidx).zfill(3)}_temp'
split_instance_output_dir.mkdir(parents=True, exist_ok=True)

query_instance = X[pidx:pidx+1]
instance_permitted_range = cf.get_local_permitted_range(
    dfXy, query_instance, features_to_vary, D.categorical_cols, continuous_cols, cf.progressive_cols,
    config, midx, savedir=split_instance_output_dir)

##### Generate 5 Sample Local Counterfactuals for a Patient

In [ ]:
e1 = cf.generate_sample_local_cf_with_permitted_range(dfXy, dexp, query_instance, instance_permitted_range, config, CFs=3)
e1_cfdf = e1.cf_examples_list[0].final_cfs_df

In [ ]:
e1_cfdf.head(2)

### Borderline & Misclassified Local Counterfactuals

In [ ]:
ioi_df, display_cols = cf.get_instances_of_interest(
    wrapped_model, X_test, y_test, config=config, split_index=midx, 
    threshold=threshold, delta=0.2, savedir=split_instance_output_dir)

In [ ]:
ioi_df.loc[pidx][['actual','pred']]

### Generate Local Counterfactuals

In [ ]:
# pidx is set above since instance_permitted_range is dependent on it
query_instance = X[pidx:pidx+1]
df_dcf = cf.generate_diverse_cfs(
    dexp,
    query_instance, 
    config,
    midx,
    threshold=threshold,
    features_to_vary=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),
    permitted_range=instance_permitted_range,
    savedir=split_instance_output_dir
    )

df_dcf.shape

### Plot Counterfactuals Heatmaps

In [ ]:
from utils2 import counterfactuals as cf
cf.plot_local_cf_heatmap(dfXy, df_dcf, query_instance, 
                      query_idx=pidx, 
                      pred=ioi_df.loc[pidx]['pred'], 
                      actual=ioi_df.loc[pidx]['actual'], 
                      highlight_invalid=False, categorical_cols=D.categorical_cols, 
                      config=config, split_index=midx,  
                      savedir=split_instance_output_dir)

### Get Most Changed Features

In [ ]:
cf.get_most_changed_feature(
    df_dcf, query_instance, config, midx, savedir=split_instance_output_dir)

###  Sparsity and L1, L2 Distances

In [ ]:
diffs, cf_ana = cf.get_local_cf_distances(
    query_instance, df_dcf, config, midx, sort_by="L2_dist", 
    savedir=split_instance_output_dir)
diffs.head()

---

## To DOs

### Prototypical and Atypical
- Prototypical (most representative)
- Atypical (deviating from standard/common)

### Sufficiency

A sufficient feature  change is one that can cause the outcome change by itself.


In [ ]:
all_features = dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),

most_changed_features_list = most_changed_features.index.to_list()
confirmed_sufficient_features = ['SSA_R', 'SSA_L', 'DL_L']
forced_timeout_features = ['HBA1C',  'DEC_AR', 'DEC_VS', 'DEC_LTS', 'DEC_PPS', 'CMAPKNE_L', 'FEET_PCT_ASYM', 'INSULIN', 'HPN', 'HAND_PCT_ASYM',  'DSLPDMIA', 'DL_R']
# Other exception: 'SPSA_R'
# No CF exception: 'SPSA_L', 'CMAPANK_L', 'MCV_L', 

# set check_features as needed
check_features = confirmed_sufficient_features[:2]
print(check_features)

df_s = cf.check_sufficiency(
    dexp,
    query_instance,
    check_features=check_features,
    maxiterations=5,
    permitted_range=instance_permitted_range,
    )
df_s

### Necessity

A necessary feature change is one that must be altered; without it, no counterfactual achieves the desired outcome.

In [ ]:
from utils2 import counterfactuals as cf
check_features = ['CAS', 'NS']
df_ns = cf.check_necessity(
    dexp,
    query_instance,
    all_features=dfXy.columns.drop(['SEX', 'Confirmed_Binary_DPN']).to_list(),
    maxiterations=500,
    total_CFs=2,
    permitted_range=instance_permitted_range,
    nrepeats=5,
    verbose=False
    )
df_ns